In [16]:
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import os
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler

# Test chuyển đổi hình ảnh

In [17]:

selected_columns = ['Flow Duration','Fwd Packets Length Total',
                    'Fwd Packet Length Max',
                    'Fwd Packet Length Min', 'Fwd Packet Length Mean',
                    'Flow Bytes/s', 'Flow Packets/s',
                    'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 
                    'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
                    'Fwd IAT Min',  'Bwd IAT Mean', 'Bwd IAT Std',
                    'Bwd IAT Max', 'Bwd IAT Min', 'Bwd PSH Flags',
                    'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
                    'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
                    'Packet Length Std', 'Packet Length Variance', 
                    'Avg Packet Size', 'Avg Fwd Segment Size',
                    'Subflow Fwd Packets', 'Subflow Fwd Bytes',
                    'Fwd Seg Size Min',
                    'Idle Mean', 'Idle Std', 'Idle Max', 'Label']

len(selected_columns)

37

In [18]:
df = pd.read_csv('data/full_data/balanced_dataset.csv')
df = df[selected_columns]

In [19]:
# Hàm phân chia các ảnh thành 3 danh sách theo tỷ lệ 60%-20%-20%
def split_df(df):

    # 1. Lấy ngẫu nhiên 60% dữ liệu cho tập thứ nhất
    df_60 = df.sample(frac=0.6, random_state=42)

    # 2. Loại bỏ những dòng đã chọn từ DataFrame ban đầu
    df_remaining = df.drop(df_60.index)

    # 3. Lấy ngẫu nhiên 20% dữ liệu (trên phần dữ liệu còn lại) cho tập thứ hai
    df_20_1 = df_remaining.sample(frac=0.5, random_state=42)

    # 4. Phần còn lại chính là 20% cuối cùng cho tập thứ ba
    df_20_2 = df_remaining.drop(df_20_1.index)

    return df_60, df_20_1, df_20_2

In [20]:

def convert(df_normalized_splited, label, num, named):
    # Chuyển mỗi dòng thành ma trận ảnh (giả sử 16x16 pixels)
    image_size = (6, 6)

    # lưu ảnh tập train
    i = 1
    for row in df_normalized_splited.values:

        image_array = np.array(row).reshape(image_size)  # hiện tại chọn 75 đặc trưng nên chọn reshape với kích thước 5x15 = 75
        image_array = np.nan_to_num(image_array)  # Thay thế các giá trị không hợp lệ bằng 0
        image = Image.fromarray((image_array * 255).astype(np.uint8)) # nhân 255 để chuyển sang giá trị RGB
        image = image.convert("RGB")
        
        image = image.resize((224, 224))  # Chuyển đổi kích thước hình ảnh thành 224x224 (thường train model dùng kích thước này)
        image.save(f"data/images/{named}/{label}/{str(i)}.png")

        i = i + 1
        if i > num:
            break


def setup_to_convert(df_normalized, label):
    os.makedirs(f'data/images/train/{label}', exist_ok=True)
    os.makedirs(f'data/images/valid/{label}', exist_ok=True)
    os.makedirs(f'data/images/test/{label}', exist_ok=True)

    # tổng số lượng ảnh train + valid + test của mỗi nhãn
    n = 500
    train_n = int(n * 0.6)  
    valid_n = int(n * 0.2)  
    test_n = int(n * 0.2)

    # chia data thành các tập train, valid, test
    train, valid, test = split_df(df_normalized)

    convert(train, label, train_n, 'train')
    convert(valid, label, valid_n, 'valid')
    convert(test, label, test_n, 'test')

In [21]:
# df_shifted = temp - temp.min() + 1  # Đảm bảo tất cả giá trị >= 1
# df_log_scaled = np.log1p(df_shifted)

# df_log_scaled.replace([np.inf, -np.inf], np.nan, inplace=True)  # Thay giá trị vô hạn bằng NaN
# df_log_scaled.fillna(df_log_scaled.median(), inplace=True)  # Điền giá trị NaN bằng giá trị trung vị

# # # Kiểm tra giá trị NaN
# # print("Số lượng NaN:", df_log_scaled.isna().sum().sum())

# # # Kiểm tra giá trị vô hạn
# # print("Có giá trị vô hạn không?", np.isinf(df_log_scaled.values).any())

# print("Giá trị lớn nhất trong dữ liệu:", df_log_scaled.max().max())
# print("Giá trị nhỏ nhất trong dữ liệu:", df_log_scaled.min().min())

In [22]:
# Nhóm dữ liệu theo cột 'Label'
grouped = df.groupby('Label')

## Tạo dictionary để lưu các DataFrame tương ứng với từng nhãn
dfs = {label: group for label, group in grouped}

for label, df_label in dfs.items():
    df_drop_label = df_label.drop(columns=['Label'])
    
    df_drop_label.replace([np.inf, -np.inf], np.nan, inplace=True)  # Thay giá trị vô hạn bằng NaN
    df_drop_label.fillna(df_drop_label.median(), inplace=True)  # Điền giá trị NaN bằng giá trị trung vị

    # data_features = np.log1p(df_drop_label + 1)

    df_shifted = df_drop_label - df_drop_label.min() + 1  # Đảm bảo tất cả giá trị >= 1
    df_log_scaled = np.log1p(df_shifted)


    data_standardized = (df_log_scaled - df_log_scaled.mean()) / df_log_scaled.std()
    data_normalized = data_standardized

    # # Khởi tạo bộ chuẩn hóa Z-score
    # standardscaler = StandardScaler()

    # # Chuẩn hóa dữ liệu
    # standardized_data = standardscaler.fit_transform(df_drop_label)

    # # log
    # df_shifted = standardized_data - standardized_data.min() + 1  # Đảm bảo tất cả giá trị >= 1
    # df_log_scaled = np.log1p(df_shifted + 1)

    


    # robust_scaler = RobustScaler()
    # robust_scaler_array = robust_scaler.fit_transform(df_log_scaled)
    # # Chuyển lại thành DataFrame, giữ nguyên tên cột
    # robust_scaler_df = pd.DataFrame(robust_scaler_array, columns=df_drop_label.columns)

    # powertransformer_scaler = PowerTransformer(method='yeo-johnson')  # Phù hợp với dữ liệu có cả giá trị âm và dương
    # powertransformer_scaler_array = powertransformer_scaler.fit_transform(robust_scaler_df)
    # powertransformer_scaler_df = pd.DataFrame(powertransformer_scaler_array, columns=df_drop_label.columns)

    # # Chuẩn hóa về khoảng [0,1]
    # min_max_scaler = MinMaxScaler(feature_range=(0, 1))
    # min_max_df = pd.DataFrame(min_max_scaler.fit_transform(powertransformer_scaler_df), columns=powertransformer_scaler_df.columns)

    # # Chuẩn hóa về khoảng [0,255]
    # df_normalized = min_max_df * 255

    # # Ép kiểu về số nguyên (tùy vào ứng dụng, có thể giữ float)
    # df_normalized_int = df_normalized.astype(int)

    setup_to_convert(data_normalized, label)


# Lưu ảnh đầu tiên
# cv2.imwrite("attack_image.png", images[0])